In [4]:
import json
import time
import urllib.parse
import urllib.request
import urllib.error
from dataclasses import dataclass
from datetime import datetime, timezone
from typing import Any, Dict, List, Optional, Tuple


# =============================
# 0) Free, no-key endpoints
# =============================

NOMINATIM_SEARCH_URL = "https://nominatim.openstreetmap.org/search"
OPEN_METEO_FORECAST_URL = "https://api.open-meteo.com/v1/forecast"

# IMPORTANT: Use a real contact email to reduce 403 risk.
# Many environments get blocked if this looks generic.
CONTACT_EMAIL = "your.name@yourdomain.com"

USER_AGENT = f"WeatherAgentDemo/1.2 (contact: {CONTACT_EMAIL})"

# Be polite to Nominatim (rate limiting)
NOMINATIM_MIN_SECONDS_BETWEEN_CALLS = 1.1
_last_nominatim_call_ts: float = 0.0


# =============================
# 1) Severe weather thresholds
# =============================

THRESHOLDS = {
    "wind_kph_warning": 50.0,
    "wind_kph_critical": 80.0,

    "precip_mm_next_hour_warning": 10.0,
    "precip_mm_next_hour_critical": 25.0,
    "precip_mm_daily_warning": 30.0,
    "precip_mm_daily_critical": 60.0,

    "temp_c_high_warning": 35.0,
    "temp_c_high_critical": 38.0,
    "temp_c_low_warning": -5.0,
    "temp_c_low_critical": -10.0,

    "lookahead_hours": 24,
    "lookahead_days": 3,
}


# =============================
# 2) Data structures
# =============================

@dataclass(frozen=True)
class SevereAlert:
    severity: str  # "warning" | "critical"
    kind: str      # "wind" | "precip" | "heat" | "cold"
    when: str
    message: str
    facts: Dict[str, Any]


# =============================
# 3) HTTP helper (with basic retry)
# =============================

def _http_get_json(
    url: str,
    params: Dict[str, Any],
    headers: Optional[Dict[str, str]] = None,
    timeout_s: int = 30,
    retries: int = 2,
) -> Any:
    qs = urllib.parse.urlencode(params)
    full_url = f"{url}?{qs}"
    req = urllib.request.Request(full_url, headers=headers or {}, method="GET")

    last_err: Optional[Exception] = None
    for attempt in range(retries + 1):
        try:
            with urllib.request.urlopen(req, timeout=timeout_s) as resp:
                raw = resp.read().decode("utf-8")
                return json.loads(raw)
        except urllib.error.HTTPError as e:
            last_err = e
            # Retry on common transient statuses
            if e.code in (429, 500, 502, 503, 504) and attempt < retries:
                time.sleep(1.0 + attempt * 1.5)
                continue
            # 403 is typically policy/blocking; do not retry endlessly
            raise
        except Exception as e:
            last_err = e
            if attempt < retries:
                time.sleep(1.0 + attempt * 1.5)
                continue
            raise

    raise RuntimeError(f"HTTP request failed: {last_err!r}")


# =============================
# 4) Geocoding (Nominatim)
# =============================

def get_lat_long(city: str, state: str, country: str = "USA") -> Tuple[float, float]:
    global _last_nominatim_call_ts

    if not CONTACT_EMAIL or "@" not in CONTACT_EMAIL:
        raise ValueError("Set CONTACT_EMAIL to a real email (required for Nominatim etiquette).")

    # Rate limit
    now = time.time()
    sleep_s = NOMINATIM_MIN_SECONDS_BETWEEN_CALLS - (now - _last_nominatim_call_ts)
    if sleep_s > 0:
        time.sleep(sleep_s)

    q = f"{city}, {state}, {country}".strip().strip(",")

    params = {
        "q": q,
        "format": "jsonv2",
        "limit": 1,
        "addressdetails": 1,
        # Some deployments use this to contact heavy users; harmless for light use
        "email": CONTACT_EMAIL,
    }
    headers = {
        "User-Agent": USER_AGENT,
        "From": CONTACT_EMAIL,
        "Accept": "application/json",
        "Accept-Language": "en",
    }

    try:
        data = _http_get_json(NOMINATIM_SEARCH_URL, params=params, headers=headers, timeout_s=30, retries=1)
    except urllib.error.HTTPError as e:
        if e.code == 403:
            raise RuntimeError(
                "Nominatim returned 403 Forbidden. "
                "This often happens when the User-Agent/From/contact is missing/generic, "
                "or the runtime IP is blocked. Set CONTACT_EMAIL to a real email and try again."
            ) from e
        raise
    finally:
        _last_nominatim_call_ts = time.time()

    if not data:
        raise ValueError(f"Location not found: {q!r}")

    return (float(data[0]["lat"]), float(data[0]["lon"]))


# =============================
# 5) Extended forecast (Open-Meteo)
# =============================

def get_extended_weather_forecast(lat: float, lon: float, forecast_days: int = 7) -> Dict[str, Any]:
    params = {
        "latitude": lat,
        "longitude": lon,
        "timezone": "auto",
        "forecast_days": int(forecast_days),
        "daily": ",".join([
            "temperature_2m_max",
            "temperature_2m_min",
            "precipitation_sum",
            "wind_speed_10m_max",
        ]),
        "hourly": ",".join([
            "temperature_2m",
            "precipitation",
            "wind_speed_10m",
        ]),
        "current_weather": "true",
    }
    headers = {"Accept": "application/json"}
    return _http_get_json(OPEN_METEO_FORECAST_URL, params=params, headers=headers, timeout_s=30, retries=2)


# =============================
# 6) Severe-weather evaluation
# =============================

def _severity_from_value(value: float, warning: float, critical: float) -> Optional[str]:
    if value >= critical:
        return "critical"
    if value >= warning:
        return "warning"
    return None


def evaluate_severe_weather(payload: Dict[str, Any], thresholds: Dict[str, Any]) -> List[SevereAlert]:
    alerts: List[SevereAlert] = []

    hourly = payload.get("hourly") or {}
    daily = payload.get("daily") or {}
    current = payload.get("current_weather") or {}

    cur_temp = current.get("temperature")
    cur_wind = current.get("windspeed")
    cur_time = current.get("time") or datetime.now(timezone.utc).isoformat()

    if isinstance(cur_wind, (int, float)):
        sev = _severity_from_value(float(cur_wind), thresholds["wind_kph_warning"], thresholds["wind_kph_critical"])
        if sev:
            alerts.append(SevereAlert(severity=sev, kind="wind", when=str(cur_time),
                                      message=f"High wind now: {float(cur_wind):.1f} km/h",
                                      facts={"wind_kph": float(cur_wind), "source": "current_weather"}))

    if isinstance(cur_temp, (int, float)):
        hot = _severity_from_value(float(cur_temp), thresholds["temp_c_high_warning"], thresholds["temp_c_high_critical"])
        if hot:
            alerts.append(SevereAlert(severity=hot, kind="heat", when=str(cur_time),
                                      message=f"High temperature now: {float(cur_temp):.1f}°C",
                                      facts={"temp_c": float(cur_temp), "source": "current_weather"}))
        if float(cur_temp) <= thresholds["temp_c_low_critical"]:
            alerts.append(SevereAlert(severity="critical", kind="cold", when=str(cur_time),
                                      message=f"Extreme cold now: {float(cur_temp):.1f}°C",
                                      facts={"temp_c": float(cur_temp), "source": "current_weather"}))
        elif float(cur_temp) <= thresholds["temp_c_low_warning"]:
            alerts.append(SevereAlert(severity="warning", kind="cold", when=str(cur_time),
                                      message=f"Low temperature now: {float(cur_temp):.1f}°C",
                                      facts={"temp_c": float(cur_temp), "source": "current_weather"}))

    h_times = hourly.get("time") or []
    h_prec = hourly.get("precipitation") or []
    h_wind = hourly.get("wind_speed_10m") or []
    h_temp = hourly.get("temperature_2m") or []

    lookahead_hours = int(thresholds.get("lookahead_hours", 24))
    n_h = min(lookahead_hours, len(h_times), len(h_prec), len(h_wind), len(h_temp))

    for i in range(n_h):
        when = str(h_times[i])

        try:
            p = float(h_prec[i])
            sev = _severity_from_value(p, thresholds["precip_mm_next_hour_warning"], thresholds["precip_mm_next_hour_critical"])
            if sev:
                alerts.append(SevereAlert(severity=sev, kind="precip", when=when,
                                          message=f"Heavy precipitation: {p:.1f} mm in an hour",
                                          facts={"precip_mm": p, "window": "1h", "source": "hourly"}))
        except Exception:
            pass

        try:
            w = float(h_wind[i])
            sev = _severity_from_value(w, thresholds["wind_kph_warning"], thresholds["wind_kph_critical"])
            if sev:
                alerts.append(SevereAlert(severity=sev, kind="wind", when=when,
                                          message=f"High wind: {w:.1f} km/h",
                                          facts={"wind_kph": w, "source": "hourly"}))
        except Exception:
            pass

        try:
            t = float(h_temp[i])
            sev_hot = _severity_from_value(t, thresholds["temp_c_high_warning"], thresholds["temp_c_high_critical"])
            if sev_hot:
                alerts.append(SevereAlert(severity=sev_hot, kind="heat", when=when,
                                          message=f"High temperature: {t:.1f}°C",
                                          facts={"temp_c": t, "source": "hourly"}))
            if t <= thresholds["temp_c_low_critical"]:
                alerts.append(SevereAlert(severity="critical", kind="cold", when=when,
                                          message=f"Extreme cold: {t:.1f}°C",
                                          facts={"temp_c": t, "source": "hourly"}))
            elif t <= thresholds["temp_c_low_warning"]:
                alerts.append(SevereAlert(severity="warning", kind="cold", when=when,
                                          message=f"Low temperature: {t:.1f}°C",
                                          facts={"temp_c": t, "source": "hourly"}))
        except Exception:
            pass

    d_times = daily.get("time") or []
    d_p_sum = daily.get("precipitation_sum") or []
    d_tmax = daily.get("temperature_2m_max") or []
    d_tmin = daily.get("temperature_2m_min") or []
    d_wmax = daily.get("wind_speed_10m_max") or []

    lookahead_days = int(thresholds.get("lookahead_days", 3))
    n_d = min(lookahead_days, len(d_times), len(d_p_sum), len(d_tmax), len(d_tmin), len(d_wmax))

    for i in range(n_d):
        when = str(d_times[i])

        try:
            p = float(d_p_sum[i])
            sev = _severity_from_value(p, thresholds["precip_mm_daily_warning"], thresholds["precip_mm_daily_critical"])
            if sev:
                alerts.append(SevereAlert(severity=sev, kind="precip", when=when,
                                          message=f"High daily precipitation: {p:.1f} mm (day total)",
                                          facts={"precip_mm": p, "window": "day", "source": "daily"}))
        except Exception:
            pass

        try:
            w = float(d_wmax[i])
            sev = _severity_from_value(w, thresholds["wind_kph_warning"], thresholds["wind_kph_critical"])
            if sev:
                alerts.append(SevereAlert(severity=sev, kind="wind", when=when,
                                          message=f"High daily max wind: {w:.1f} km/h",
                                          facts={"wind_kph": w, "source": "daily"}))
        except Exception:
            pass

        try:
            tmax = float(d_tmax[i])
            sev_hot = _severity_from_value(tmax, thresholds["temp_c_high_warning"], thresholds["temp_c_high_critical"])
            if sev_hot:
                alerts.append(SevereAlert(severity=sev_hot, kind="heat", when=when,
                                          message=f"High daily max temperature: {tmax:.1f}°C",
                                          facts={"temp_c": tmax, "metric": "daily_max", "source": "daily"}))
        except Exception:
            pass

        try:
            tmin = float(d_tmin[i])
            if tmin <= thresholds["temp_c_low_critical"]:
                alerts.append(SevereAlert(severity="critical", kind="cold", when=when,
                                          message=f"Extreme daily min temperature: {tmin:.1f}°C",
                                          facts={"temp_c": tmin, "metric": "daily_min", "source": "daily"}))
            elif tmin <= thresholds["temp_c_low_warning"]:
                alerts.append(SevereAlert(severity="warning", kind="cold", when=when,
                                          message=f"Low daily min temperature: {tmin:.1f}°C",
                                          facts={"temp_c": tmin, "metric": "daily_min", "source": "daily"}))
        except Exception:
            pass

    def _rank(sev: str) -> int:
        return {"warning": 1, "critical": 2}.get(sev, 0)

    dedup: Dict[Tuple[str, str, str], SevereAlert] = {}
    for a in alerts:
        key = (a.kind, a.when, a.message.split(":")[0])
        prev = dedup.get(key)
        if (prev is None) or (_rank(a.severity) > _rank(prev.severity)):
            dedup[key] = a

    out = list(dedup.values())
    out.sort(key=lambda a: (-_rank(a.severity), a.when, a.kind))
    return out


# =============================
# 7) Demo for Gaithersburg, Maryland
# =============================

def main() -> None:
    lat_long = get_lat_long("Gaithersburg", "Maryland")
    print("lat_long = get_lat_long('Gaithersburg','Maryland') ->", lat_long)

    lat, lon = lat_long
    forecast = get_extended_weather_forecast(lat, lon, forecast_days=7)

    print("\nExtended forecast (summary)")
    print("timezone:", forecast.get("timezone"))
    if forecast.get("current_weather"):
        print("current_weather:", forecast["current_weather"])

    daily = forecast.get("daily", {}) or {}
    times = daily.get("time", []) or []
    tmax = daily.get("temperature_2m_max", []) or []
    tmin = daily.get("temperature_2m_min", []) or []
    p_sum = daily.get("precipitation_sum", []) or []
    w_max = daily.get("wind_speed_10m_max", []) or []

    for i in range(min(len(times), 7)):
        print(
            f"{times[i]} | max {tmax[i]}°C, min {tmin[i]}°C | "
            f"precip {p_sum[i]} mm | wind_max {w_max[i]} km/h"
        )

    alerts = evaluate_severe_weather(forecast, THRESHOLDS)
    print("\nSevere-weather threshold alerts")
    if not alerts:
        print("No threshold breaches detected in lookahead windows.")
    else:
        for a in alerts:
            print(f"[{a.severity.upper()}] {a.when} | {a.kind} | {a.message}")


if __name__ == "__main__":
    main()

lat_long = get_lat_long('Gaithersburg','Maryland') -> (39.1399187, -77.1929215)

Extended forecast (summary)
timezone: America/New_York
current_weather: {'time': '2026-03-04T00:00', 'interval': 900, 'temperature': 4.0, 'windspeed': 6.2, 'winddirection': 173, 'is_day': 0, 'weathercode': 51}
2026-03-04 | max 13.3°C, min 4.0°C | precip 0.8 mm | wind_max 10.0 km/h
2026-03-05 | max 13.7°C, min 6.9°C | precip 0.77 mm | wind_max 12.9 km/h
2026-03-06 | max 19.1°C, min 8.1°C | precip 3.0 mm | wind_max 18.7 km/h
2026-03-07 | max 24.3°C, min 7.4°C | precip 5.0 mm | wind_max 24.2 km/h
2026-03-08 | max 16.5°C, min 7.3°C | precip 0.0 mm | wind_max 22.4 km/h
2026-03-09 | max 21.2°C, min 9.0°C | precip 0.0 mm | wind_max 13.5 km/h
2026-03-10 | max 26.1°C, min 13.5°C | precip 0.0 mm | wind_max 26.1 km/h

Severe-weather threshold alerts
No threshold breaches detected in lookahead windows.
